# Project 8: Food Delivery Operations Analytics
**Industry:** FoodTech / Logistics  |  **Student:** Daksh Agarwal  |  **Student ID:** 2026b03110270
**Course:** Python, NumPy & Pandas – Five-Day Engineering Student Training (First Year, First Semester)

**Business scenario:** Evaluate delivery times, order outcomes, restaurant performance and city-level operations to identify service improvements.

**Dataset:** `08_food_delivery.csv` (2,000 orders, 4 cities, 5 restaurants). It is a synthetic teaching dataset, so findings apply to this data only.

**How to run in Google Colab:** run the cells from top to bottom. When asked, upload `08_food_delivery.csv`.

**Tasks covered:** 109 to 123 (all 15 tasks), plus extra analysis and a limitations section.

## Step 0: Setup and data loading

In [1]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# Find the dataset: same folder, ./data folder, or upload (Google Colab)
DATA_PATH = "08_food_delivery.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join("data", "08_food_delivery.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join("..", "data", "08_food_delivery.csv")
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Please upload 08_food_delivery.csv")
        files.upload()
        DATA_PATH = "08_food_delivery.csv"
    except ImportError:
        raise FileNotFoundError("Place 08_food_delivery.csv in this folder or in a data folder")

# Folder for saved result tables (file handling)
os.makedirs("outputs", exist_ok=True)
print("Using dataset:", DATA_PATH)

Using dataset: ../data/08_food_delivery.csv


## Task 109: Load and inspect the order dataset

In [2]:
df = pd.read_csv(DATA_PATH)

print("Rows, columns:", df.shape)
print()
df.info()

Rows, columns: (2000, 10)

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Order_ID           2000 non-null   str    
 1   City               2000 non-null   str    
 2   Restaurant         2000 non-null   str    
 3   Order_Value        2000 non-null   float64
 4   Distance_km        2000 non-null   float64
 5   Delivery_Time_Min  1972 non-null   float64
 6   Order_Status       2000 non-null   str    
 7   Payment_Mode       2000 non-null   str    
 8   Order_Hour         2000 non-null   int64  
 9   Record_Source      2000 non-null   str    
dtypes: float64(3), int64(1), str(6)
memory usage: 156.4 KB


In [3]:
df.head()

,Order_ID,City,Restaurant,Order_Value,Distance_km,Delivery_Time_Min,Order_Status,Payment_Mode,Order_Hour,Record_Source
0,FD00000,Noida,Green Bowl,588.00,2.6,32.0,Delivered,Card,10,Original
1,FD00001,Delhi,Pizza Point,496.68,5.3,36.0,Delivered,Cash,22,Original
2,FD00002,Faridabad,Pizza Point,419.41,6.0,42.0,Delivered,Cash,18,Original
3,FD00003,Noida,Pizza Point,195.48,5.0,40.0,Delivered,UPI,19,Original
4,FD00004,Faridabad,Cafe 24,621.70,2.5,15.0,Delivered,Cash,22,Original


In [4]:
df.describe()

,Order_Value,Distance_km,Delivery_Time_Min,Order_Hour
count,2000.000000,2000.000000,1972.000000,2000.000000
mean,451.894200,5.125550,36.066937,15.526500
std,166.914499,1.931048,11.405162,4.729004
min,80.000000,0.500000,10.000000,8.000000
25%,343.555000,3.800000,28.750000,11.000000
50%,449.880000,5.200000,36.000000,16.000000
75%,559.657500,6.400000,43.000000,20.000000
max,944.270000,10.400000,78.000000,23.000000


In [5]:
# Unique values of each text column (uses a dictionary and a set)
text_columns = ["City", "Restaurant", "Order_Status", "Payment_Mode", "Record_Source"]
unique_values = {col: sorted(set(df[col])) for col in text_columns}
for col, values in unique_values.items():
    print(col, "->", values)

City -> ['Delhi', 'Faridabad', 'Gurugram', 'Noida']
Restaurant -> ['Cafe 24', 'Green Bowl', 'Pizza Point', 'Spice Hub', 'Urban Bites']
Order_Status -> ['Cancelled', 'Delayed', 'Delivered']
Payment_Mode -> ['Card', 'Cash', 'UPI']
Record_Source -> ['Original', 'Synthetic']


## Task 110: Find missing delivery times and decide how to handle them
**Decision:** Only a small share of rows are missing, so deleting them would waste data, and delivery times are slightly right-skewed, so the mean would be pulled up by very long deliveries. I therefore fill each missing value with the **median delivery time of the same city**, and keep a flag column so the filled rows stay traceable.

In [6]:
missing_counts = df.isnull().sum()
print(missing_counts)

missing_time = df["Delivery_Time_Min"].isnull().sum()
print()
print("Missing delivery times:", missing_time)
print("Share of all orders: ", round(missing_time / len(df) * 100, 2), "%")

Order_ID              0
City                  0
Restaurant            0
Order_Value           0
Distance_km           0
Delivery_Time_Min    28
Order_Status          0
Payment_Mode          0
Order_Hour            0
Record_Source         0
dtype: int64

Missing delivery times: 28
Share of all orders:  1.4 %


In [7]:
# Which order statuses have the missing values?
df[df["Delivery_Time_Min"].isnull()]["Order_Status"].value_counts()

Order_Status
Delivered    25
Cancelled     2
Delayed       1
Name: count, dtype: int64

In [8]:
# Handle: flag first, then fill with the city median
df["Delivery_Time_Filled"] = df["Delivery_Time_Min"].isnull()
city_median = df.groupby("City")["Delivery_Time_Min"].transform("median")
df["Delivery_Time_Min"] = df["Delivery_Time_Min"].fillna(city_median)

print("Missing after cleaning:", df["Delivery_Time_Min"].isnull().sum())
print("Rows flagged as filled:", df["Delivery_Time_Filled"].sum())

Missing after cleaning: 0
Rows flagged as filled: 28


## Task 111: Check duplicate Order_ID values

In [9]:
dup_ids = df["Order_ID"].duplicated().sum()
dup_rows = df.duplicated().sum()
print("Duplicate Order_ID values:", dup_ids)
print("Fully duplicated rows:    ", dup_rows)

# Remove duplicates if any exist (keeps the first record of each Order_ID)
rows_before = len(df)
df = df.drop_duplicates(subset="Order_ID", keep="first").reset_index(drop=True)
print("Rows before:", rows_before, "| rows after:", len(df))

Duplicate Order_ID values: 0
Fully duplicated rows:     0
Rows before: 2000 | rows after: 2000


## Task 112: Total orders and order-status percentages

In [10]:
total_orders = len(df)
print("Total orders:", total_orders)

status_counts = df["Order_Status"].value_counts()
status_pct = (df["Order_Status"].value_counts(normalize=True) * 100).round(2)

status_table = pd.DataFrame({"Orders": status_counts, "Percentage": status_pct})
status_table

Total orders: 2000


,Orders,Percentage
Order_Status,,
Delivered,1642,82.10
Cancelled,185,9.25
Delayed,173,8.65


## Task 113: Average delivery time by city

In [11]:
city_time = df.groupby("City")["Delivery_Time_Min"].mean().round(2).sort_values()
city_time.to_frame("Avg_Delivery_Time_Min")

,Avg_Delivery_Time_Min
City,
Delhi,35.65
Noida,36.00
Faridabad,36.07
Gurugram,36.54


## Task 114: Five restaurants with the highest total order value

In [12]:
restaurant_total = df.groupby("Restaurant")["Order_Value"].sum().round(2)
top5_restaurants = restaurant_total.nlargest(5)
top5_restaurants.to_frame("Total_Order_Value")

,Total_Order_Value
Restaurant,
Pizza Point,202159.89
Green Bowl,189623.29
Urban Bites,181800.76
Spice Hub,173260.41
Cafe 24,156944.05


## Task 115: Average order value by restaurant

In [13]:
restaurant_avg = (
    df.groupby("Restaurant")["Order_Value"]
      .agg(Orders="count", Avg_Order_Value="mean")
      .round(2)
      .sort_values("Avg_Order_Value", ascending=False)
)
restaurant_avg

,Orders,Avg_Order_Value
Restaurant,,
Cafe 24,340,461.60
Spice Hub,380,455.95
Urban Bites,402,452.24
Pizza Point,452,447.26
Green Bowl,426,445.13


## Task 116: Identify peak ordering hours

In [14]:
orders_per_hour = df["Order_Hour"].value_counts().sort_index()
peak_hours = orders_per_hour.nlargest(3)

print("Top 3 peak hours (hour of day: orders)")
for hour, count in peak_hours.items():
    print(f"  {hour:02d}:00 -> {count} orders")

hour_summary = df.groupby("Order_Hour").agg(
    Orders=("Order_ID", "count"),
    Total_Order_Value=("Order_Value", "sum"),
    Avg_Delivery_Time=("Delivery_Time_Min", "mean"),
).round(2)
hour_summary["Order_Share_Pct"] = (hour_summary["Orders"] / total_orders * 100).round(2)
hour_summary.sort_values("Orders", ascending=False)

Top 3 peak hours (hour of day: orders)
  10:00 -> 161 orders
  13:00 -> 153 orders
  21:00 -> 148 orders


,Orders,Total_Order_Value,Avg_Delivery_Time,Order_Share_Pct
Order_Hour,,,,
10,161,73589.68,36.21,8.05
13,153,69009.66,38.07,7.65
21,148,67639.87,37.51,7.40
18,147,66594.45,34.70,7.35
9,136,62145.96,34.38,6.80
23,136,59190.38,32.68,6.80
8,132,63043.56,35.67,6.60
15,125,54865.69,34.23,6.25
20,125,58211.38,38.50,6.25


## Task 117: Compare delivery time across order-status categories

In [15]:
status_time = (
    df.groupby("Order_Status")["Delivery_Time_Min"]
      .agg(["count", "mean", "median", "min", "max", "std"])
      .round(2)
)
status_time

,count,mean,median,min,max,std
Order_Status,,,,,,
Cancelled,185,36.30,37.0,10.0,74.0,10.81
Delayed,173,42.69,44.0,13.0,78.0,11.68
Delivered,1642,35.34,35.0,10.0,78.0,11.12


## Task 118: Cancellation and delay rates by city

In [16]:
status_by_city = pd.crosstab(df["City"], df["Order_Status"], normalize="index") * 100
rates_by_city = status_by_city[["Cancelled", "Delayed"]].round(2)
rates_by_city.columns = ["Cancellation_Rate_Pct", "Delay_Rate_Pct"]
rates_by_city.sort_values("Delay_Rate_Pct", ascending=False)

,Cancellation_Rate_Pct,Delay_Rate_Pct
City,,
Delhi,11.11,9.22
Faridabad,8.21,8.89
Noida,8.55,8.33
Gurugram,9.36,8.09


## Task 119: Median and standard deviation of delivery time (NumPy)

In [17]:
times = df["Delivery_Time_Min"].to_numpy()

median_time = np.median(times)
std_time = np.std(times, ddof=1)       # sample standard deviation
mean_time = np.mean(times)

print("Mean delivery time  :", round(mean_time, 2), "min")
print("Median delivery time:", round(median_time, 2), "min")
print("Std deviation       :", round(std_time, 2), "min")

# Cross-check with Pandas (should match)
print("Pandas std check    :", round(df["Delivery_Time_Min"].std(), 2))

Mean delivery time  : 36.06 min
Median delivery time: 36.0 min
Std deviation       : 11.33 min
Pandas std check    : 11.33


## Task 120: Identify unusually long deliveries using IQR
Rule: a delivery is an outlier if it is above **Q3 + 1.5 x IQR** (or below **Q1 - 1.5 x IQR**).

In [18]:
q1 = np.percentile(times, 25)
q3 = np.percentile(times, 75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Q1:", q1, "| Q3:", q3, "| IQR:", iqr)
print("Lower bound:", round(lower_bound, 2), "| Upper bound:", round(upper_bound, 2))

df["Long_Delivery_Outlier"] = df["Delivery_Time_Min"] > upper_bound
long_deliveries = df[df["Long_Delivery_Outlier"]]
print("Unusually long deliveries:", len(long_deliveries),
      f"({len(long_deliveries) / total_orders * 100:.2f}% of orders)")

Q1: 29.0 | Q3: 43.0 | IQR: 14.0
Lower bound: 8.0 | Upper bound: 64.0
Unusually long deliveries: 15 (0.75% of orders)


In [19]:
# Where do the long deliveries happen?
print(long_deliveries["City"].value_counts())
print()
print(long_deliveries["Order_Status"].value_counts())
long_deliveries.sort_values("Delivery_Time_Min", ascending=False).head(10)

City
Faridabad    6
Gurugram     4
Noida        3
Delhi        2
Name: count, dtype: int64

Order_Status
Delivered    10
Delayed       4
Cancelled     1
Name: count, dtype: int64


,Order_ID,City,Restaurant,Order_Value,Distance_km,Delivery_Time_Min,Order_Status,Payment_Mode,Order_Hour,Record_Source,Delivery_Time_Filled,Long_Delivery_Outlier
701,FD00701,Faridabad,Green Bowl,671.62,4.2,78.0,Delivered,Cash,9,Synthetic,False,True
651,FD00651,Faridabad,Green Bowl,256.51,2.9,78.0,Delivered,Card,19,Synthetic,False,True
1901,FD01901,Faridabad,Spice Hub,488.38,5.9,78.0,Delayed,Cash,17,Synthetic,False,True
1031,FD01031,Gurugram,Urban Bites,691.11,4.4,78.0,Delivered,UPI,16,Synthetic,False,True
723,FD00723,Noida,Pizza Point,392.78,5.7,75.0,Delayed,Card,14,Synthetic,False,True
454,FD00454,Noida,Spice Hub,497.76,6.9,75.0,Delivered,UPI,20,Original,False,True
718,FD00718,Delhi,Cafe 24,283.90,2.4,74.0,Cancelled,UPI,15,Synthetic,False,True
825,FD00825,Delhi,Urban Bites,629.15,6.7,72.0,Delivered,UPI,10,Synthetic,False,True
604,FD00604,Gurugram,Pizza Point,250.84,5.1,72.0,Delayed,Card,20,Synthetic,False,True
57,FD00057,Faridabad,Spice Hub,479.34,3.5,71.0,Delivered,Card,10,Original,False,True


## Task 121: Create delivery-time bands (Fast, Standard, Delayed)
**Assumption (business rule):** Fast = up to 30 min, Standard = 31 to 45 min, Delayed = more than 45 min. These cut-offs sit close to the data's quartiles (Q1 about 29, Q3 about 43), so each band is meaningful.

In [20]:
band_labels = ("Fast", "Standard", "Delayed")     # tuple of labels

conditions = [
    df["Delivery_Time_Min"] <= 30,
    (df["Delivery_Time_Min"] > 30) & (df["Delivery_Time_Min"] <= 45),
    df["Delivery_Time_Min"] > 45,
]
df["Delivery_Band"] = np.select(conditions, list(band_labels), default="Unknown")

# Cross-check using pd.cut: both methods must agree
band_cut = pd.cut(df["Delivery_Time_Min"], bins=[0, 30, 45, np.inf], labels=list(band_labels))
print("np.select and pd.cut agree:", (df["Delivery_Band"] == band_cut.astype(str)).all())

band_table = df["Delivery_Band"].value_counts().reindex(list(band_labels)).to_frame("Orders")
band_table["Percentage"] = (band_table["Orders"] / total_orders * 100).round(2)
band_table

np.select and pd.cut agree: True


,Orders,Percentage
Delivery_Band,,
Fast,624,31.2
Standard,984,49.2
Delayed,392,19.6


In [21]:
# Bands by city (percentage of each city's orders)
(pd.crosstab(df["City"], df["Delivery_Band"], normalize="index")[list(band_labels)] * 100).round(2)

Delivery_Band,Fast,Standard,Delayed
City,,,
Delhi,33.33,50.10,16.56
Faridabad,31.11,49.06,19.83
Gurugram,28.30,50.85,20.85
Noida,32.05,46.79,21.15


**Note:** The `Order_Status` column marks only about 8.65% of orders as "Delayed", while the time-based Delayed band flags more orders. The two measures differ because status is recorded by the business, while the band is a pure time rule; both are reported separately.

## Task 122: Group revenue by city and payment mode
**Assumption:** Revenue counts orders that were **Delivered or Delayed** (the customer received the food). Cancelled orders are excluded from revenue, and their lost value is shown separately.

In [22]:
revenue_df = df[df["Order_Status"] != "Cancelled"]

revenue_pivot = pd.pivot_table(
    revenue_df,
    values="Order_Value",
    index="City",
    columns="Payment_Mode",
    aggfunc="sum",
    margins=True,
    margins_name="Total",
).round(2)
revenue_pivot

Payment_Mode,Card,Cash,UPI,Total
City,,,,
Delhi,71889.12,61071.93,55821.70,188782.75
Faridabad,94261.41,84042.73,59706.66,238010.80
Gurugram,70625.43,64854.26,63067.45,198547.14
Noida,65769.59,71389.56,56213.28,193372.43
Total,302545.55,281358.48,234809.09,818713.12


In [23]:
# Lost value due to cancellations, by city
lost_value = df[df["Order_Status"] == "Cancelled"].groupby("City")["Order_Value"].sum().round(2)
print("Order value lost to cancellations:")
print(lost_value)
print("Total lost:", round(lost_value.sum(), 2))

Order value lost to cancellations:
City
Delhi        25814.22
Faridabad    22560.63
Gurugram     19446.67
Noida        17253.76
Name: Order_Value, dtype: float64
Total lost: 85075.28


## Extra analysis (beyond the required tasks)
Distance and restaurant-level delays help explain the recommendations. These show association only, not cause.

In [24]:
# How strongly do distance and delivery time move together?
distance_corr = np.corrcoef(df["Distance_km"], df["Delivery_Time_Min"])[0, 1]
print("Correlation (distance vs delivery time):", round(distance_corr, 3))

# Restaurant-level delay and cancellation rates
restaurant_rates = pd.crosstab(df["Restaurant"], df["Order_Status"], normalize="index") * 100
restaurant_rates = restaurant_rates[["Cancelled", "Delayed"]].round(2)
restaurant_rates.columns = ["Cancellation_Rate_Pct", "Delay_Rate_Pct"]
restaurant_rates["Avg_Delivery_Time"] = df.groupby("Restaurant")["Delivery_Time_Min"].mean().round(2)
restaurant_rates.sort_values("Delay_Rate_Pct", ascending=False)

Correlation (distance vs delivery time): 0.254


,Cancellation_Rate_Pct,Delay_Rate_Pct,Avg_Delivery_Time
Restaurant,,,
Green Bowl,6.81,10.80,36.48
Spice Hub,8.95,9.47,35.83
Urban Bites,9.70,8.96,36.43
Cafe 24,7.65,7.06,34.93
Pizza Point,12.61,6.86,36.39


## Task 123: Five operational recommendations
Each recommendation is built from the results above (values are calculated, not typed in).

In [25]:
# Gather the facts used in the recommendations
slowest_city = city_time.idxmax()
slowest_city_time = city_time.max()
fastest_city = city_time.idxmin()

worst_delay_city = rates_by_city["Delay_Rate_Pct"].idxmax()
worst_delay_rate = rates_by_city["Delay_Rate_Pct"].max()
worst_cancel_city = rates_by_city["Cancellation_Rate_Pct"].idxmax()
worst_cancel_rate = rates_by_city["Cancellation_Rate_Pct"].max()

peak_list = ", ".join(f"{h:02d}:00" for h in sorted(peak_hours.index))
peak_share = round(peak_hours.sum() / total_orders * 100, 1)

worst_rest = restaurant_rates["Delay_Rate_Pct"].idxmax()
worst_rest_rate = restaurant_rates["Delay_Rate_Pct"].max()

top_rest = top5_restaurants.index[0]
top_rest_share = round(top5_restaurants.iloc[0] / restaurant_total.sum() * 100, 1)

pay_revenue = revenue_pivot.loc["Total"].drop("Total")
top_pay = pay_revenue.idxmax()
top_pay_share = round(pay_revenue.max() / pay_revenue.sum() * 100, 1)

delayed_pct = band_table.loc["Delayed", "Percentage"]

recommendations = [
    f"1. Use one network-wide delivery standard: city averages are very close ({city_time.min()} to {slowest_city_time} min, "
    f"a gap of only {round(slowest_city_time - city_time.min(), 2)} min), so the problem is not one slow city. "
    f"{slowest_city} is slowest by a small margin; focus on the long-delivery tail instead, because "
    f"{delayed_pct}% of orders take over 45 min.",

    f"2. Add rider capacity at peak hours ({peak_list}): these three hours carry {peak_share}% of all orders, "
    f"which is above an even spread over the 16 active hours ({round(3/16*100,1)}%), so shortages here are most likely to hurt service.",

    f"3. Reduce delays in {worst_delay_city} (delay rate {worst_delay_rate}%) and cancellations in "
    f"{worst_cancel_city} (cancellation rate {worst_cancel_rate}%). Set city-wise targets and track them weekly.",

    f"4. Work with {worst_rest}, which has the highest delay rate ({worst_rest_rate}%), to shorten "
    f"preparation and handover time, while protecting the top-revenue partner {top_rest} "
    f"({top_rest_share}% of order value).",

    f"5. Set about {int(upper_bound)} min as an alert limit: deliveries above it are statistical outliers "
    f"({len(long_deliveries)} orders). Assign these to a review queue, and keep {top_pay} payments smooth, "
    f"as they bring the most revenue ({top_pay_share}%).",
]

for rec in recommendations:
    print(rec)
    print()

1. Use one network-wide delivery standard: city averages are very close (35.65 to 36.54 min, a gap of only 0.89 min), so the problem is not one slow city. Gurugram is slowest by a small margin; focus on the long-delivery tail instead, because 19.6% of orders take over 45 min.

2. Add rider capacity at peak hours (10:00, 13:00, 21:00): these three hours carry 23.1% of all orders, which is above an even spread over the 16 active hours (18.8%), so shortages here are most likely to hurt service.

3. Reduce delays in Delhi (delay rate 9.22%) and cancellations in Delhi (cancellation rate 11.11%). Set city-wise targets and track them weekly.

4. Work with Green Bowl, which has the highest delay rate (10.8%), to shorten preparation and handover time, while protecting the top-revenue partner Pizza Point (22.4% of order value).

5. Set about 64 min as an alert limit: deliveries above it are statistical outliers (15 orders). Assign these to a review queue, and keep Card payments smooth, as they b

## Save deliverable tables (file handling)

In [26]:
# 1) City performance table
city_performance = pd.DataFrame({
    "Orders": df.groupby("City")["Order_ID"].count(),
    "Avg_Delivery_Time_Min": city_time,
    "Cancellation_Rate_Pct": rates_by_city["Cancellation_Rate_Pct"],
    "Delay_Rate_Pct": rates_by_city["Delay_Rate_Pct"],
    "Revenue": revenue_df.groupby("City")["Order_Value"].sum().round(2),
}).sort_values("Revenue", ascending=False)

# 2) Restaurant ranking
restaurant_ranking = pd.DataFrame({
    "Total_Order_Value": restaurant_total,
    "Avg_Order_Value": restaurant_avg["Avg_Order_Value"],
    "Delay_Rate_Pct": restaurant_rates["Delay_Rate_Pct"],
}).sort_values("Total_Order_Value", ascending=False)
restaurant_ranking.insert(0, "Rank", range(1, len(restaurant_ranking) + 1))

city_performance.to_csv("outputs/city_performance.csv")
restaurant_ranking.to_csv("outputs/restaurant_ranking.csv")
hour_summary.sort_values("Orders", ascending=False).to_csv("outputs/peak_hour_summary.csv")
band_table.to_csv("outputs/delay_bands.csv")
status_time.to_csv("outputs/delay_by_status.csv")
revenue_pivot.to_csv("outputs/revenue_city_payment.csv")

# Recommendations as a text file
with open("outputs/recommendations.txt", "w", encoding="utf-8") as f:
    for rec in recommendations:
        f.write(rec + "\n\n")

print("Saved files:", sorted(os.listdir("outputs")))

Saved files: ['city_performance.csv', 'delay_bands.csv', 'delay_by_status.csv', 'peak_hour_summary.csv', 'recommendations.txt', 'restaurant_ranking.csv', 'revenue_city_payment.csv']


In [27]:
city_performance

,Orders,Avg_Delivery_Time_Min,Cancellation_Rate_Pct,Delay_Rate_Pct,Revenue
City,,,,,
Faridabad,585,36.07,8.21,8.89,238010.80
Gurugram,470,36.54,9.36,8.09,198547.14
Noida,468,36.00,8.55,8.33,193372.43
Delhi,477,35.65,11.11,9.22,188782.75


In [28]:
restaurant_ranking

,Rank,Total_Order_Value,Avg_Order_Value,Delay_Rate_Pct
Restaurant,,,,
Pizza Point,1,202159.89,447.26,6.86
Green Bowl,2,189623.29,445.13,10.80
Urban Bites,3,181800.76,452.24,8.96
Spice Hub,4,173260.41,455.95,9.47
Cafe 24,5,156944.05,461.60,7.06


## Assumptions and limitations
- The data is **synthetic** (a teaching dataset), so results describe this dataset and not real-world operations.
- Missing delivery times (about 1.4% of rows) were filled with the city median; this is an assumption and slightly reduces variation.
- Delay bands use business cut-offs of 30 and 45 minutes, which are assumptions and can be changed.
- Revenue excludes cancelled orders; if the business counts cancelled orders differently, totals will change.
- The dataset has no dates, so weekday or seasonal patterns cannot be studied.
- Relationships such as distance vs delivery time are **associations only; correlation does not prove causation**.